# Install and load dependencies

In [ ]:
# ! pip install -q evaluate bitsandbytes
# ! pip install -qU datasets

In [ ]:
import os
import torch
import evaluate
from tqdm.auto import tqdm
from datasets import load_dataset
from torch.utils.data import DataLoader

from transformers import (
    AutoTokenizer,
    DataCollatorWithPadding,
    get_scheduler,
    AutoModelForSequenceClassification,
)

# Config

In [ ]:
checkpoint = "bert-base-uncased"
num_train_epochs = 4
train_bs = 32
gacc_steps = 2                      # gradient accumulation steps
lr_scheduler_type = "linear"
lr = 5e-5
max_grad_norm = 1.0                 # gradient clipping threshold
use_8bit_adam = True

# Load dataset
- [MRCP Dataset](https://aclanthology.org/)

In [ ]:
raw_datasets = load_dataset("glue", "mrpc")

In [ ]:
print("====== overall split and scehma")
display(raw_datasets)
print("====== a single record")
display(raw_datasets["train"][0])
print("====== what does each label value mean?")
display(raw_datasets["train"].features)
print("====== saved dataset file structure (Apache Arrows File)")
os.listdir("/root/.cache/huggingface/datasets/")

# Preprocess the dataset

In [ ]:
def tokenize_function(example, tokenizer):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)
    
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

tokenized_datasets = raw_datasets.map(
    tokenize_function,
    batched=True,
    remove_columns = ['sentence1', 'sentence2', 'idx'],
    fn_kwargs = {'tokenizer': tokenizer}
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## Test the prep pipeline

In [ ]:
samples = tokenized_datasets["train"][:8]
samples = {k: v for k, v in samples.items()}
print("========= lengths before collating/padding")
print("input_ids: ", [len(x) for x in samples["input_ids"]])
print("attention_mask: ", [len(x) for x in samples["attention_mask"]])
print("========= lengths after collating/padding")
batch = data_collator(samples)
{k: v.shape for k, v in batch.items()}

## Some additional steps
- These are handled by the Hugging Face `Trainer` automatically

In [ ]:
# rename "label" to "labels"
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
# set format to torch.Tensor
tokenized_datasets.set_format("torch")

# Create Dataloaders

In [ ]:
train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=train_bs, collate_fn=data_collator
)
eval_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=train_bs, collate_fn=data_collator
)

In [ ]:
batch = next(iter(train_dataloader))
{k: v.shape for k, v in batch.items()}

# Train

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

In [ ]:
outputs = model(**batch)
print(outputs.loss, outputs.logits.shape)

In [ ]:
# The default optimizer of the HF Trainer is AdamW
# Adam incorporates weight decay into the moving averages of gradients (L2 regularization),
    # which can weaken the regularization effect.
# AdamW decouples this, applying weight decay directly to the parameters,
    # leading to more effective, independent regularization.

if use_8bit_adam:
    # Can be slightly slower in some cases due to extra quant/dequant overhead.
    try:
        import bitsandbytes as bnb
        optimizer = bnb.optim.AdamW8bit(model.parameters(), lr=lr)
        print("Using bitsandbytes AdamW8bit optimizer.")
    except Exception as e:
        print(f"Could not use bitsandbytes 8-bit optimizer (falling back to torch AdamW). Reason: {e}")
        from torch.optim import AdamW
        optimizer = AdamW(model.parameters(), lr=lr)
else:
    from torch.optim import AdamW
    optimizer = AdamW(model.parameters(), lr=lr)

In [ ]:
num_update_steps_per_epoch = (len(train_dataloader) + gacc_steps - 1) // gacc_steps
num_training_steps = num_train_epochs * num_update_steps_per_epoch

lr_scheduler = get_scheduler(
    lr_scheduler_type,
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

print("Train batches per epoch:", len(train_dataloader))
print("Gradient accumulation steps:", gacc_steps)
print("Optimizer updates per epoch:", num_update_steps_per_epoch)
print("Total optimizer updates (num_training_steps):", num_training_steps)

In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device);

In [ ]:
bf16_supported = torch.cuda.is_bf16_supported()
amp_dtype = torch.bfloat16 if bf16_supported else torch.float16
use_amp = (device.type == "cuda")
# GradScaler is mainly needed for fp16; for bf16 it's typically not needed
scaler = torch.amp.GradScaler(device.type, enabled=(use_amp and amp_dtype == torch.float16))
print(f"AMP enabled: {use_amp} | amp_dtype: {amp_dtype} | scaler enabled: {scaler.is_enabled()}")

In [ ]:
metric = evaluate.load("glue", "mrpc")
train_progress_bar = tqdm(total=num_training_steps, desc="Optimizer Updates")

for epoch in range(num_train_epochs):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(train_dataloader):
        batch = {k: v.to(device) for k, v in batch.items()}

        # autocast forward
        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
            outputs = model(**batch)
            loss = outputs.loss
            loss = loss / gacc_steps

        # backward with scaler
        scaler.scale(loss).backward()

        # Do an optimizer step every gacc_steps (or at end)
        is_accum_step = ((step + 1) % gacc_steps == 0) or ((step + 1) == len(train_dataloader))
        if is_accum_step:
            # Gradient clipping (must unscale first when using scaler)
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)

            scaler.step(optimizer)
            scaler.update()
            lr_scheduler.step()
            optimizer.zero_grad(set_to_none=True)

            train_progress_bar.update(1)

    model.eval()
    eval_progress_bar = tqdm(total=len(eval_dataloader), desc=f"Eval (epoch {epoch+1})", leave=True)

    for batch in eval_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
                outputs = model(**batch)

        predictions = torch.argmax(outputs.logits, dim=-1)
        metric.add_batch(predictions=predictions, references=batch["labels"])
        eval_progress_bar.update(1)

    results = metric.compute()
    print(f"\nEpoch {epoch+1}/{num_train_epochs} - MRPC metrics:", results)

print("Done.")